# Home Credit Risk Analysis

## Dataset: bureau.csv

### Objetivo 

Analizar la estructura, calidad y relaciones del dataset bureau para comprender el historial crediticio externo de los clientes y su integración dentro del modelo dimensional.

---

### Contexto del negocio 

Este dataset contiene información histórica de créditos reportados por otras entidades financieras. 

Cada registro representa un crédito externo asociado a un cliente. 

La información permite conocer el comportamiento financiero previo de los solicitantes y constituye una de las principales fuentes para la evaluación del riesgo crediticio. 

---

### Rol dentro de la arquitectura 

applicaction_train 
        │
        ▼
      bureau
        │
        ▼
  bureau_balance

Este dataset será utilizado para construir métricas históricas de comportamiento crediticio dentro de las capas Silver, Intermediate y Gold. 

# 1. Importación de librerías 

Se cargan las librerías necesarias para el análisis exploratorio de datos. 

In [2]:
import pandas as pd 
import numpy as np 

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# 2. Carga del Dataset 

Este dataset representa el historial crediticio externo de los clientes reportado por entidades financieras.

Cada registro corresponde a un crédito histórico asociado a un cliente y contiene información sobre estado, deuda, mora y características del producto financiero.

In [ ]:
df = pd.read_csv("../../data/raw/bureau.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

## Observaciones Iniciales

El dataset contiene **1.716.428 registros** y corresponde al historial crediticio externo de los clientes reportado por entidades financieras.

Cada registro representa un crédito histórico asociado a un cliente identificado mediante la clave `SK_ID_CURR`.

La alta cantidad de registros evidencia que un mismo cliente puede poseer múltiples obligaciones financieras reportadas por diferentes entidades.

### Información contenida

El dataset incorpora principalmente información relacionada con:

- Historial crediticio.
- Estado de los créditos.
- Tipo de producto financiero.
- Montos otorgados.
- Deuda pendiente.
- Mora histórica.
- Exposición financiera.

### Rol dentro del ecosistema Home Credit

Este dataset complementa la información principal contenida en `application_train`, aportando contexto histórico sobre el comportamiento financiero previo de los clientes.

Su principal objetivo es enriquecer el perfil de riesgo crediticio mediante información proveniente de fuentes externas.

# 3. Vista preliminar de los datos 

Se inspeccionan los primeros registros para comprender la estructura general y el tipo de información almacenada.

In [ ]:
df.head()

# 4. Estructura General 

Se analiza la composición del dataset: 

- Cantidad de columnas 
- Tipos de datos 
- Presencia de valores nulos 
- Distribución general de atributos 

In [ ]:
df.info()

# 5. Identificación de la Clave Primaria 

El dataset contiene dis identificadores principales: 

- `SK_ID_BUREAU`
- `SK_ID_CURR`

El objetivo de esta sección es determinar: 

1. La clave primaria que identifica de manera única cada registro. 
2. La clave foránea que permite relacionar el dataset con otras entidades del dominio Home Credit. 
3. La cardinalidad existente entre clientes y créditos históricos. 

La correcta identificación de estas claves es fundamental para: 

- Garantizar la integridad de los datos. 
- Diseñar las relaciones del modelo dimensional. 
- Construir procesos de integración en las capas Bronze, Silver e Intermediate. 
- Definir reglas de calidad y validación de datos. 

In [ ]:
total_rows = len(df)

bureau_unique = df["SK_ID_BUREAU"].nunique()
customer_unique = df["SK_ID_CURR"].nunique()

bureau_duplicates = df["SK_ID_BUREAU"].duplicated().sum()

print("Resgistros: ", total_rows)

print("SK_ID_BUREAU únicos:", bureau_unique)
print("SK_ID_CURR únicos:", customer_unique)
print("Duplicados SK_ID_BUREAU:", bureau_duplicates)

## Resultados 

A partir del análisis realizado de identificó que: 

| Métrica | Valor |
|----------|---------|
| Registros totales | 1,716,428 |
| SK_ID_BUREAU únicos | 1,716,428 |
| SK_ID_CURR únicos | 305,811 |
| Duplicados SK_ID_BUREAU | 0 |

### Clave Primaria (PK) 

La columna `SK_ID_BUREAU` identifica de manera única cada registro del dataset.

No se detectaron valores duplicados, por lo que puede considerarse una clave primaria válida.

### Clave Foránea (FK)

La columna `SK_ID_CURR` representa el identificador del cliente y permite relacionar este dataset con `application_train`.

### Cardinalidad

application_train (1)

↓

bureau (N)

Un cliente puede estar asociado a múltiples créditos históricos reportados por entidades financieras externas.

### Interpretación de Negocio

El dataset no almacena información a nivel de cliente, sino a nivel de crédito histórico.

Por esta razón, un mismo cliente puede aparecer múltiples veces dentro del dataset, mientras que cada crédito reportado posee un identificador único.

In [ ]:
avg_credits = len(df) / df["SK_ID_CURR"].nunique()

print(f"Promedio de créditos por cliente: {avg_credits:.2f}")

### Relación Cliente - Créditos

Se observó que el dataset contiene 1,716,428 registros asociados a 305,811 clientes únicos.

Esto indica que cada cliente posee en promedio aproximadamente 5.6 créditos reportados en entidades financieras externas.

### Interpretación

La existencia de múltiples registros por cliente confirma una relación uno a muchos entre:

application_train (Cliente)

↓

bureau (Créditos históricos)

Esta característica convierte a bureau en una de las principales fuentes para construir indicadores agregados de comportamiento financiero.

# 6. Análisis de Calidad de Datos

La calidad de los datos constituye un aspecto fundamental dentro de la arquitectura Lakehouse, ya que determina la confiabilidad de los procesos analíticos posteriores.

En esta sección se evalúan:

- Valores faltantes
- Completitud de atributos
- Posibles problemas de calidad
- Impacto sobre las futuras capas Silver e Intermediate

Los hallazgos obtenidos permitirán definir reglas de validación y transformación para los pipelines ELT.

In [ ]:
null_percentage = (
    df.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

null_percentage.head(20)

In [ ]:
null_df = (
    null_percentage
    .reset_index()
)

null_df.columns = ["Column", "Null_Percentage"]

null_df.head(20)

In [ ]:
high_nulls = null_percentage[null_percentage > 50]

medium_nulls = null_percentage[
    (null_percentage > 20)
    & (null_percentage <= 50)
]

low_nulls = null_percentage[
    (null_percentage > 0)
    & (null_percentage <= 20)
]

print("Columnas > 50% nulos:", len(high_nulls))
print("Columnas entre 20% y 50%:", len(medium_nulls))
print("Columnas < 20%:", len(low_nulls))

## Hallazgos sobre la Calidad de los Datos

El análisis de completitud evidencia que la mayoría de los atributos del dataset presentan una calidad adecuada y altos niveles de disponibilidad de información.

Sin embargo, se identificaron algunas variables financieras con porcentajes significativos de valores faltantes.

### Variables con mayor porcentaje de nulos

| Columna | % Nulos |
|----------|----------|
| AMT_ANNUITY | 71.47% |
| AMT_CREDIT_MAX_OVERDUE | 65.51% |
| DAYS_ENDDATE_FACT | 36.92% |
| AMT_CREDIT_SUM_LIMIT | 34.48% |
| AMT_CREDIT_SUM_DEBT | 15.01% |

### Interpretación de Negocio

Las columnas con mayores niveles de nulidad corresponden principalmente a información financiera específica de determinados tipos de crédito.

La ausencia de valores no necesariamente representa un error de captura, sino que puede indicar:

- Créditos donde no aplica una anualidad.
- Créditos sin historial de mora registrado.
- Créditos sin línea de crédito disponible.
- Créditos que aún permanecen activos y no poseen fecha de finalización efectiva.

### Impacto en la Arquitectura Lakehouse

#### Bronze

Los datos serán almacenados sin modificaciones para preservar la trazabilidad del origen.

#### Silver

Se deberán definir reglas específicas para:

- Validar la completitud de atributos financieros.
- Diferenciar nulos de negocio frente a nulos de calidad.
- Evaluar estrategias de imputación o exclusión según el caso.

#### Gold y Diamond

Las métricas derivadas de deuda, mora y exposición crediticia deberán considerar explícitamente estos porcentajes de ausencia para evitar sesgos analíticos.

# 7. Detección de Registros Duplicados

Se evalúa la existencia de registros completamente duplicados dentro del dataset.

La presencia de duplicados puede afectar:

- Indicadores financieros
- Métricas de exposición crediticia
- Cálculos agregados
- Integridad del modelo dimensional

Por esta razón se verifica la unicidad de los registros antes de su incorporación a la arquitectura Lakehouse.

In [ ]:
df.duplicated().sum()

## Hallazgos

No se identificaron registros completamente duplicados dentro del dataset.

### Conclusión

El dataset presenta una adecuada integridad estructural y puede ser considerado una fuente confiable para la construcción de indicadores financieros históricos.

Esta validación refuerza la consistencia observada previamente en la clave primaria `SK_ID_BUREAU`.

# 8.  Variables de Negocio Relevantes

A diferencia de application_train, este dataset no contiene una variable objetivo.

Su propósito principal es complementar la información histórica de comportamiento crediticio de los clientes mediante atributos financieros provenientes de entidades externas.

In [14]:
important_cols = [
    "CREDIT_ACTIVE",
    "CREDIT_TYPE",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "CREDIT_DAY_OVERDUE"
]

df[important_cols].head()

,CREDIT_ACTIVE,CREDIT_TYPE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,CREDIT_DAY_OVERDUE
0,Closed,Consumer credit,91323.0,0.0,0
1,Active,Credit card,225000.0,171342.0,0
2,Active,Consumer credit,464323.5,NaN,0
3,Active,Credit card,90000.0,NaN,0
4,Active,Consumer credit,2700000.0,NaN,0


## Interpretación de las Variables Financieras

La muestra analizada permite identificar información clave sobre el comportamiento histórico de los clientes frente a otras entidades financieras.

### CREDIT_ACTIVE

Representa el estado actual del crédito reportado.

Algunos valores observados:

- Active
- Closed

Esta variable permitirá construir indicadores relacionados con:

- Créditos vigentes.
- Créditos finalizados.
- Distribución del portafolio crediticio.

---

### CREDIT_TYPE

Describe el tipo de producto financiero otorgado al cliente.

Ejemplos observados:

- Consumer credit
- Credit card

Esta información será útil para segmentar la exposición financiera según el tipo de obligación adquirida.

---

### AMT_CREDIT_SUM

Corresponde al monto total del crédito otorgado.

Ejemplos identificados:

- 91,323
- 225,000
- 464,323.5
- 2,700,000

Esta variable será utilizada para calcular indicadores de exposición financiera histórica.

---

### AMT_CREDIT_SUM_DEBT

Representa la deuda pendiente asociada al crédito.

Se observa que algunos registros contienen valores nulos, lo que puede indicar:

- Créditos completamente saldados.
- Información no reportada por la entidad financiera.
- Créditos donde la deuda actual no aplica.

Esta variable será fundamental para construir métricas de endeudamiento y riesgo financiero.

---

### CREDIT_DAY_OVERDUE

Indica la cantidad de días de mora registrados para el crédito.

En los registros observados no se identifican días de mora, lo que sugiere que estos créditos se encuentran al día.

Sin embargo, el análisis completo del dataset permitirá identificar clientes con historial de incumplimiento y niveles de riesgo superiores.

---

## Relevancia para el Modelo Analítico

Las variables analizadas constituyen la base para la construcción de indicadores relacionados con:

- Endeudamiento total.
- Exposición financiera.
- Historial crediticio.
- Mora acumulada.
- Riesgo financiero histórico.

Estos indicadores serán generados en las capas Silver, Intermediate y Gold para alimentar posteriormente los procesos analíticos y de convergencia en Diamond.

# 9. Variables Categóricas

Se identifican las variables categóricas presentes en el dataset para comprender los dominios de negocio disponibles y las posibles dimensiones analíticas.

In [15]:
categorical_columns = df.select_dtypes(include=["object"])

categorical_columns.columns.tolist()

C:\Users\ferna\AppData\Local\Temp\ipykernel_14640\1644596.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"])


['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']

In [16]:
for col in categorical_columns.columns:
    print(col, "-", df[col].nunique())

CREDIT_ACTIVE - 4
CREDIT_CURRENCY - 4
CREDIT_TYPE - 15


## Cardinalidad de las Variables Categóricas

| Variable | Categorías Únicas |
|-----------|------------------|
| CREDIT_ACTIVE | 4 |
| CREDIT_CURRENCY | 4 |
| CREDIT_TYPE | 15 |

### Análisis de Cardinalidad

Las variables categóricas identificadas presentan una cardinalidad relativamente baja, lo que facilita su utilización en procesos analíticos y de modelado dimensional.

#### CREDIT_ACTIVE (4 categorías)

Representa el estado actual del crédito reportado.

Su baja cardinalidad permite clasificar fácilmente los créditos según su situación financiera actual.

Posibles categorías observadas:

- Active
- Closed
- Sold
- Bad debt

Esta variable será fundamental para la construcción de indicadores de riesgo y exposición crediticia.

---

#### CREDIT_CURRENCY (4 categorías)

Representa la moneda utilizada para registrar la obligación financiera.

La baja cantidad de categorías sugiere un entorno financiero relativamente homogéneo y facilita los procesos posteriores de estandarización monetaria.

---

#### CREDIT_TYPE (15 categorías)

Describe el producto financiero asociado al crédito.

Aunque presenta una cardinalidad superior a las demás variables categóricas, continúa siendo manejable para procesos analíticos.

Esta variable permitirá segmentar el comportamiento histórico de los clientes según:

- Créditos de consumo.
- Tarjetas de crédito.
- Hipotecas.
- Créditos para vehículos.
- Otros productos financieros.

---

### Implicaciones para el Modelo Dimensional

Estas variables constituyen candidatas naturales para dimensiones de negocio que permitirán:

- Analizar créditos por estado.
- Segmentar clientes según tipo de producto financiero.
- Construir indicadores históricos de exposición financiera.
- Generar métricas agregadas para las capas Gold y Diamond.

Debido a su baja cardinalidad, no se identifican riesgos significativos asociados a explosión de categorías o problemas de modelado.

# 10. Conclusiones Técnicas

## Resumen Ejecutivo

El dataset `bureau.csv` contiene información histórica de créditos reportados por entidades financieras externas y constituye una de las principales fuentes para enriquecer el perfil de riesgo de los clientes dentro del ecosistema Home Credit.

A diferencia de `application_train`, que representa solicitudes de crédito, este dataset almacena el historial financiero previo de los clientes y proporciona contexto adicional sobre su comportamiento crediticio.

---

## Métricas Generales

| Métrica | Valor |
|----------|---------|
| Registros | 1,716,428 |
| Clientes únicos | 305,811 |
| Clave primaria | SK_ID_BUREAU |
| Clave foránea | SK_ID_CURR |
| Duplicados | 0 |

---

## Calidad de Datos

El análisis de completitud evidenció una calidad general adecuada para los atributos principales del dataset.

Las columnas con mayor porcentaje de valores faltantes fueron:

| Columna | % Nulos |
|----------|----------|
| AMT_ANNUITY | 71.47% |
| AMT_CREDIT_MAX_OVERDUE | 65.51% |
| DAYS_ENDDATE_FACT | 36.92% |
| AMT_CREDIT_SUM_LIMIT | 34.48% |
| AMT_CREDIT_SUM_DEBT | 15.01% |

La mayoría de las columnas críticas para la integración y el análisis financiero presentan porcentajes de nulidad cercanos a cero.

---

## Granularidad

Cada registro representa un crédito histórico asociado a un cliente.

Por lo tanto:

1 registro = 1 crédito reportado

Esta granularidad permite construir indicadores financieros históricos y métricas agregadas por cliente.

---

## Relaciones Identificadas

### Relación con application_train

application_train (1)

↓

bureau (N)

Un cliente puede poseer múltiples créditos históricos registrados por diferentes entidades financieras.

---

### Relación con bureau_balance

bureau (1)

↓

bureau_balance (N)

Cada crédito histórico puede contener múltiples registros de seguimiento mensual.

---

## Variables de Negocio Relevantes

Las variables financieras más importantes identificadas fueron:

- CREDIT_ACTIVE
- CREDIT_TYPE
- AMT_CREDIT_SUM
- AMT_CREDIT_SUM_DEBT
- CREDIT_DAY_OVERDUE

Estas variables permitirán construir indicadores relacionados con:

- Endeudamiento.
- Mora histórica.
- Exposición financiera.
- Créditos activos.
- Créditos cerrados.
- Comportamiento crediticio acumulado.

---

## Variables Categóricas

Se identificaron tres variables categóricas principales:

| Variable | Categorías |
|-----------|-----------|
| CREDIT_ACTIVE | 4 |
| CREDIT_CURRENCY | 4 |
| CREDIT_TYPE | 15 |

La baja cardinalidad observada facilita la construcción de dimensiones analíticas y procesos de agregación.

---

## Rol dentro del Modelo Dimensional

El dataset será utilizado para construir la tabla:

### FACT_BUREAU_CREDIT

Esta entidad complementará la información principal de solicitudes de crédito y permitirá incorporar métricas históricas del comportamiento financiero de los clientes.

---

## Rol dentro de la Arquitectura Lakehouse

bureau.csv

↓ Bronze (ingesta cruda)

↓ Silver (validación y calidad)

↓ Intermediate (agregaciones por cliente)

↓ Gold (indicadores de negocio)

↓ Diamond (convergencia corporativa)
